# Data exploration for ML part of the project

## Pre requisite fot this notebook
Check the ML section of the readme at the root of the project to have explanation to install the virtualenv for this notebook to run.  

When it's done: Click on select Kernel
* Ctrl + Shift + P
* Select: "Enter interpreter path ..."
* Select the .venv

## Python imports

In [4]:
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import r2_score, mean_absolute_error

## Presentation of the ML part of the project
* Here we are going to compute a simple model to predict the price of an house or flat sale in France.
* The goal is not to have good prediction here but simply to put a model in a FastApi Docker container.
* I want to avoid doing another "TODO List" app.
* In order to predict the price target our model will use only very simple feature (remember we are doing this for fun):
    * Date of sale
    * Position (longitude and latitude)
    * Type (house or flat)
    * Surface area

## Get data to train ML model
* Download CSV from and save it in directory "data"
* Go to https://explore.data.gouv.fr/fr/immobilier?onglet=tableau&filtre=tous
* Enter a county name (for the example I used county Haute-Garonne)
* Then click on "Télécharger les données filtrées" (Download filtred data)

## Data investigation and data cleanning

In [14]:
df = pd.read_csv("../../data/dvf.csv")
df.head()

C:\Users\thoma\AppData\Local\Temp\ipykernel_18632\1402111326.py:1: DtypeWarning: Columns (0: numero_disposition, 1: valeur_fonciere, 2: adresse_numero, 3: code_postal, 4: code_commune, 5: code_departement, 6: ancien_code_commune, 7: ancien_nom_commune, 8: ancien_id_parcelle, 9: numero_volume, 10: lot1_numero, 11: lot1_surface_carrez, 12: lot2_numero, 13: lot2_surface_carrez, 14: lot3_numero, 15: lot3_surface_carrez, 16: lot4_numero, 17: lot4_surface_carrez, 18: lot5_numero, 19: lot5_surface_carrez, 20: nombre_lots, 21: code_type_local, 22: surface_reelle_bati, 23: nombre_pieces_principales, 24: surface_terrain, 25: longitude, 26: latitude) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../../data/dvf.csv")


,id_mutation,date_mutation,numero_disposition,nature_mutation,valeur_fonciere,adresse_numero,adresse_suffixe,adresse_nom_voie,adresse_code_voie,code_postal,...,surface_reelle_bati,nombre_pieces_principales,code_nature_culture,nature_culture,code_nature_culture_speciale,nature_culture_speciale,surface_terrain,longitude,latitude,section_prefixe
0,2025-371476,2025-09-23,2,Echange,1.0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.407092,43.60168,844AC
1,2025-371476,2025-09-23,2,Echange,1.0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.407092,43.60168,844AC
2,2025-371476,2025-09-23,2,Echange,1.0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.407092,43.60168,844AC
3,2025-371476,2025-09-23,1,Echange,1.0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.407092,43.60168,844AC
4,2025-371476,2025-09-23,1,Echange,1.0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.407092,43.60168,844AC


In [16]:
# Many data that needs to be cleaned
df.shape

(409366, 41)

In [17]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 409366 entries, 0 to 409365
Data columns (total 41 columns):
 #   Column                        Non-Null Count   Dtype 
---  ------                        --------------   ----- 
 0   id_mutation                   409366 non-null  str   
 1   date_mutation                 409366 non-null  str   
 2   numero_disposition            409366 non-null  object
 3   nature_mutation               409366 non-null  str   
 4   valeur_fonciere               407740 non-null  object
 5   adresse_numero                288027 non-null  object
 6   adresse_suffixe               26357 non-null   str   
 7   adresse_nom_voie              406554 non-null  str   
 8   adresse_code_voie             406560 non-null  str   
 9   code_postal                   406546 non-null  object
 10  code_commune                  409366 non-null  object
 11  nom_commune                   409366 non-null  str   
 12  code_departement              409366 non-null  object
 13  ancien_cod

Sorry it's in french but let me translate for you the esentials:  

|FR|EN|
|---|---|
|nature_mutation|transaction_type|
|Vente|Sale|
|type_local|property_type|
|Maison|House|
|Appartement|Flat|
|valeur_fonciere|Price|
|surface_relle_bati|living_area in square meters|
|latitude|latitude|
|longitude|longitude|
|nombre_lots|number_of_lots|
|id_mutation|transaction_id|
|nombre_pieces_principales|room_count|
|code_postal|postal_code|
|code_commune|commune_code (city_code)|

In [19]:
columns_to_keep = [
    'id_mutation',
    'code_postal',
    'date_mutation',
    #'code_commune',
    'nombre_pieces_principales',
    'nombre_lots',
    'nature_mutation',
    'type_local',
    'valeur_fonciere',
    #'latitude',
    #'longitude',
    'surface_reelle_bati'

]

df_cleaned = df[columns_to_keep].copy()

df_cleaned = df_cleaned.drop_duplicates(subset='id_mutation', keep='first')
df_cleaned = df_cleaned.dropna()
#df_cleaned = df_cleaned.drop_duplicates()

df_cleaned = df_cleaned[df_cleaned['nature_mutation'] == 'Vente']
df_cleaned = df_cleaned[df_cleaned['nombre_lots'] == 1]

#df_cleaned = df_cleaned[df_cleaned['type_local'].isin(['Maison','Appartement'])]

df_cleaned = df_cleaned[df_cleaned['type_local'] == 'Appartement']

df_cleaned['date_mutation'] = pd.to_datetime(df_cleaned['date_mutation'], errors='coerce')
df_cleaned = df_cleaned.dropna(subset=['date_mutation'])

df_cleaned['valeur_fonciere'] = pd.to_numeric(df_cleaned['valeur_fonciere'], errors='coerce')
df_cleaned['surface_reelle_bati'] = pd.to_numeric(df_cleaned['surface_reelle_bati'], errors='coerce')
#df_cleaned['latitude'] = pd.to_numeric(df_cleaned['latitude'], errors='coerce')
#df_cleaned['longitude'] = pd.to_numeric(df_cleaned['longitude'], errors='coerce')
df_cleaned['nombre_pieces_principales'] = pd.to_numeric(df_cleaned['nombre_pieces_principales'], errors='coerce')

df_cleaned = df_cleaned[df_cleaned['nombre_pieces_principales'] > 1]
df_cleaned = df_cleaned[df_cleaned['surface_reelle_bati'] > 30]
df_cleaned = df_cleaned[df_cleaned['valeur_fonciere'] > 10000]

columns_to_keep2 = [
    #'id_mutation',
    'code_postal',
    'date_mutation',
    #'code_commune',
    'nombre_pieces_principales',
    #'nombre_lots',
    #'nature_mutation',
    #'type_local',
    'valeur_fonciere',
    #'latitude',
    #'longitude',
    'surface_reelle_bati'
]

df_cleaned = df_cleaned[columns_to_keep2]

df_cleaned.head()

,code_postal,date_mutation,nombre_pieces_principales,valeur_fonciere,surface_reelle_bati
48,31400.0,2025-01-02,2.0,135000.0,47.0
102,31500.0,2025-01-07,3.0,476290.0,110.0
134,31000.0,2025-01-09,3.0,287000.0,65.0
138,31000.0,2025-01-13,2.0,175000.0,96.0
181,31150.0,2025-01-02,3.0,160000.0,62.0


In [20]:
df_cleaned.shape

(5241, 5)

In [ ]:
# Let's rename the columns in english
df_cleaned.rename(columns={
    'date_mutation': 'date_mutation',
    'code_postal': 'postal_code',
    'nombre_pieces_principales': 'room_count',
    #'nature_mutation': 'transaction_type',
    #'type_local': 'property_type',
    'valeur_fonciere': 'price',
    'surface_reelle_bati': 'living_area',
    #'latitude': 'latitude',
    #'longitude': 'longitude'
}, inplace=True)

# And let's replace the values in English in nature_mutation and type_local
#df_cleaned['transaction_type'] = df_cleaned['transaction_type'].replace('Vente', 'Sale')
#df_cleaned['property_type'] = df_cleaned['property_type'].replace('Maison', 'House')

df_cleaned.head()

,postal_code,room_count,price,living_area
48,31400.0,2.0,135000.0,47.0
102,31500.0,3.0,476290.0,110.0
134,31000.0,3.0,287000.0,65.0
138,31000.0,2.0,175000.0,96.0
181,31150.0,3.0,160000.0,62.0


In [8]:
df_cleaned.describe()

,room_count,price,living_area
count,5241.000000,5.241000e+03,5241.000000
mean,2.739363,1.712466e+05,58.274757
std,0.895064,1.072523e+05,20.516119
min,2.000000,1.500000e+04,31.000000
25%,2.000000,1.121000e+05,43.000000
50%,3.000000,1.478640e+05,55.000000
75%,3.000000,2.000000e+05,67.000000
max,30.000000,2.473382e+06,360.000000


In [9]:
df_cleaned[['postal_code','room_count','price', 'living_area']].corr()

,postal_code,room_count,price,living_area
postal_code,1.000000,0.022665,-0.150586,0.034489
room_count,0.022665,1.000000,0.345553,0.736685
price,-0.150586,0.345553,1.000000,0.521271
living_area,0.034489,0.736685,0.521271,1.000000


## Train Model

In [10]:
X = df_cleaned[['postal_code', 'room_count', 'living_area']]
y = df_cleaned['price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

In [11]:
numeric_features = ['room_count', 'living_area']
categorical_features = ['postal_code']

preprocessor = ColumnTransformer(transformers=[
    ('numeric', StandardScaler(), numeric_features),
    ('categorical', OneHotEncoder(handle_unknown='ignore'), categorical_features),
])

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LinearRegression())
])

pipeline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numeric', ...), ('categorical', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different tra

In [12]:
y_pred = pipeline.predict(X_test)

In [13]:
# For linear regression, the score is the R-squared score
pipeline.score(X_test, y_test)  

0.5514264084449741

In [14]:
r2_score(y_test, y_pred)

0.5514264084449741

In [15]:
mean_absolute_error(y_test, y_pred)

37559.791160265064

## Save the model

In [ ]:
# Model saved to backend it'll be easier to use with our Fast API backend
joblib.dump(pipeline, '../../backend/model.pkl')

['../../backend/model.pkl']